# 04 — Simulazione "OASIS-inspired" (Blocco C)

Cruscotto del Blocco C. Il lavoro pesante gira su **HPC** (simulazione → dialoghi → **judge** → report); qui si esplora e si confronta.
Risultati per esperimento in `data/processed/simulazioni/<test>/…`; giudizi e report in `data/processed/giudizi/<test>/…`.
- `test1_baseline` (taglie diverse, 5 round) · `test2_stessa_fascia_round10` (~70B, 10 round) · `test3_tag_kb` (~70B, 10 round, **tag base_kb** + **judge**).

In [ ]:
import sys
from pathlib import Path
RADICE = Path.cwd().parent
sys.path.insert(0, str(RADICE))
import pandas as pd
from IPython.display import IFrame, display
from src.simulation.scenari import SCENARI, elenco
from src.simulation.run_simulazione import carica, riepilogo, scenari_disponibili, modelli_disponibili, test_disponibili
print("Scenari:", elenco()); print("Esperimenti:", test_disponibili())

## 1) Prova locale (mock, senza HPC)

In [ ]:
from src.simulation.oasis_inspired import simula, responder_mock
from src.simulation.mappa_sim import genera
res = simula(SCENARI["siccita_darfur"], responder_mock, n_round=4, verbose=True)
genera(res, titolo="demo_locale", out_path=RADICE/"data/processed/graphs/simulazioni/demo_locale.html")
IFrame(src="../data/processed/graphs/simulazioni/demo_locale.html", width="100%", height=520)

## 2) Mappa di una run (HPC)

In [ ]:
TEST = (test_disponibili() or ["test1_baseline"])[-1]
SCEN = "siccita_darfur"
print("esperimento:", TEST, "| modelli:", modelli_disponibili(TEST, SCEN))
MOD = (modelli_disponibili(TEST, SCEN) or ["mock"])[0]
IFrame(src=f"../data/processed/simulazioni/{TEST}/{SCEN}/{MOD}/mappa.html", width="100%", height=520)

## 3) Confronto tra modelli (sintesi + mappe)

In [ ]:
TEST = (test_disponibili() or ["test1_baseline"])[-1]; SCEN = "siccita_darfur"
mods = modelli_disponibili(TEST, SCEN)
righe = [{"modello": m, **riepilogo(carica(TEST, SCEN, m))} for m in mods if carica(TEST, SCEN, m)]
display(pd.DataFrame(righe).set_index("modello") if righe else pd.DataFrame())
for m in mods:
    print(f"=== {SCEN} · {m} ==="); display(IFrame(src=f"../data/processed/simulazioni/{TEST}/{SCEN}/{m}/mappa.html", width="100%", height=480))

## 4) Dialogo tra modelli (ragionamento + tag base_kb)
Una colonna per modello: per ogni round il 💭 ragionamento, la reazione, i cambi di stato/archi (col tag `base_kb` = da dove nella KB viene la scelta) e gli eventi.

In [ ]:
from src.simulation.dialogo import genera_dialogo
TEST = (test_disponibili() or ["test1_baseline"])[-1]; SCEN = "ransomware_usa"
genera_dialogo(TEST, SCEN)
IFrame(src=f"../data/processed/simulazioni/{TEST}/{SCEN}/dialogo.html", width="100%", height=680)

## 5) Judge (coerenza) e report grafico
Un modello giudice (famiglia diversa, es. `command-r`) valuta ogni dialogo contro la KB: punteggi 0–100 + motivazione + evidenze. Il report mostra classifica, grounding oggettivo e heatmap.

In [ ]:
from src.simulation.report_giudizi import genera_report
from src.simulation.judge import carica_giudizio
TEST = (test_disponibili() or ["test1_baseline"])[-1]
try:
    genera_report(TEST)   # (ri)genera dai giudizi presenti (se ci sono)
    display(IFrame(src=f"../data/processed/giudizi/{TEST}/report.html", width="100%", height=760))
except Exception as e:
    print("Nessun giudizio ancora per", TEST, "(gira il judge su HPC).", e)